In [ ]:
! pip install scikit-survival lifelines

In [ ]:
from lifelines import WeibullFitter, CoxPHFitter, WeibullAFTFitter
from lifelines import KaplanMeierFitter, NelsonAalenFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test
from lifelines.datasets import load_rossi

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.inspection import permutation_importance

from sksurv.datasets import load_gbsg2
from sksurv.preprocessing import OneHotEncoder
from sksurv.ensemble import RandomSurvivalForest

import numpy as np
import pandas as pd

from lifelines.fitters import ParametricUnivariateFitter
import autograd.numpy as np
from autograd.scipy.stats import norm
from autograd.scipy.special import expit, logit

from matplotlib import pyplot as plt

In [ ]:
rossi = load_rossi()

[Rossi Recidivism Dataset](https://rdrr.io/cran/RcmdrPlugin.survival/man/Rossi.html)

In [ ]:
rossi.head(5)

In [ ]:
T, E = rossi.week, rossi.arrest

# Basic nonparametrics analysis

## Using Kaplan-Meier Estimator

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E)

In [ ]:
kmf.survival_function_.plot()
plt.title('Time to Repeated Crime')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

fin_aid = rossi.fin == 1

kmf_aid = KaplanMeierFitter()
ax = kmf_aid.fit(
    T[fin_aid], event_observed=E[fin_aid], label='fin aid'
    ).plot_survival_function(ax=ax)

kmf_control = KaplanMeierFitter()
ax = kmf_control.fit(
    T[~fin_aid], event_observed=E[~fin_aid], label='control'
    ).plot_survival_function(ax=ax)

plt.title('Time to Repeated Crime: Treatment vs Control')
add_at_risk_counts(kmf_aid, kmf_control, ax=ax)
plt.tight_layout()


### logrank test

In [ ]:
results = logrank_test(T[fin_aid], T[~fin_aid], E[fin_aid], E[~fin_aid], alpha=.95)

results.print_summary()

## Estimating hazard rates using Nelson-Aalen

In [ ]:
naf = NelsonAalenFitter()
naf.fit(T,event_observed=E)

In [ ]:
print(naf.cumulative_hazard_.head())
naf.plot_cumulative_hazard()

In [ ]:
naf.fit(T[fin_aid], event_observed=E[fin_aid], label="Treatment")
ax = naf.plot_cumulative_hazard(loc=slice(0, 20))

naf.fit(T[~fin_aid], event_observed=E[~fin_aid], label="Control")
naf.plot_cumulative_hazard(ax=ax, loc=slice(0, 20))

plt.title("Repeated Crime: Cumulative Hazards")

# Univariate Parametric Models

## Weibull model

In [ ]:
wf = WeibullFitter().fit(T, E)

wf.print_summary()
ax = wf.plot_cumulative_hazard()
ax.set_title("Cumulative hazard of Weibull model")

In [ ]:
ax = wf.plot_survival_function()
ax.set_title("Survival function of Weibull model")

# Semiparametrics Models

## Cox’s proportional hazard model

In [ ]:
cph = CoxPHFitter()
cph.fit(rossi, duration_col='week', event_col='arrest')

cph.print_summary()

## with penalties

In [ ]:
cph = CoxPHFitter(penalizer=0.1, l1_ratio=1.0)
cph.fit(rossi, 'week', 'arrest')
cph.print_summary()

In [ ]:
penalty = np.array([0, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5])

cph = CoxPHFitter(penalizer=penalty)
cph.fit(rossi, 'week', 'arrest')
cph.print_summary()

## Plotting the effect of varying a covariate

In [ ]:
cph = CoxPHFitter()
cph.fit(rossi, duration_col='week', event_col='arrest')

cph.plot_partial_effects_on_outcome(covariates='prio', values=[0, 2, 4, 6, 8, 10], cmap='coolwarm')

# Parametric survival models

## The Weibull AFT model

In [ ]:
aft = WeibullAFTFitter()
aft.fit(rossi, duration_col='week', event_col='arrest')

aft.print_summary(3)

In [ ]:
print(aft.median_survival_time_)
print(aft.mean_survival_time_)

### With ancillary parameters

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

times = np.arange(0, 100)
wft_model_rho = WeibullAFTFitter().fit(rossi, 'week', 'arrest', ancillary=True, timeline=times)
wft_model_rho.plot_partial_effects_on_outcome('prio', range(0, 16, 3), cmap='coolwarm', ax=ax[0])
ax[0].set_title("Modelling rho_")

wft_not_model_rho = WeibullAFTFitter().fit(rossi, 'week', 'arrest', ancillary=False, timeline=times)
wft_not_model_rho.plot_partial_effects_on_outcome('prio', range(0, 16, 3), cmap='coolwarm', ax=ax[1])
ax[1].set_title("Not modelling rho_");

# Random Survival Forest

In [ ]:
random_state=42

In [ ]:
dt = [('cens', '?'), ('time', '<f8')]
y = np.asarray(list(zip(E.astype(bool), T.astype(float))), dtype=dt)
X = rossi.iloc[:, 2:].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=random_state)

In [ ]:
y[:10]

In [ ]:
rsf = RandomSurvivalForest(
    n_estimators=1000, min_samples_split=6, min_samples_leaf=10, n_jobs=-1, random_state=random_state
)
rsf.fit(X_train, y_train)

In [ ]:
rsf.score(X_test, y_test)

In [ ]:
X_test_sorted = X_test.sort_values(by=["prio", "age"])
X_test_sel = pd.concat((X_test_sorted.head(10), X_test_sorted.tail(10)))

In [ ]:
pd.Series(rsf.predict(X_test_sel))[:10].mean()

In [ ]:
pd.Series(rsf.predict(X_test_sel))[10:].mean()

In [ ]:
surv = rsf.predict_survival_function(X_test_sel, return_array=True)

g1, g2 = surv[:10, :].mean(axis=0), surv[10:, :].mean(axis=0)

plt.step(rsf.unique_times_, g1, where='post', label='g1')
plt.step(rsf.unique_times_, g2, where='post', label='g2')

plt.ylabel("Repeated Crime Survival")
plt.xlabel("Time in weeks")
plt.legend()
plt.grid(True)

## Permutation-based Feature Importance

In [ ]:
result = permutation_importance(rsf, X_test, y_test, n_repeats=15, random_state=random_state)

In [ ]:
pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
).sort_values(by="importances_mean", ascending=False)

# Cure Models

In [ ]:
N = 1000
U = np.random.rand(N)
T = -(logit(-np.log(U) / 0.5) - np.random.exponential(2, N) - 6.00) / 0.50

E = ~np.isnan(T)
T[np.isnan(T)] = 50

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 4))

t = np.linspace(0, 40)
llf = WeibullFitter().fit(T, E, timeline=t)

t = np.linspace(0, 100)
llf = WeibullFitter().fit(T, E, timeline=t)

llf.plot_survival_function(ax=ax[0])
kmf.plot(ax=ax[0])

llf.plot_cumulative_hazard(ax=ax[1])
naf.plot(ax=ax[1])

In [ ]:
kmf = KaplanMeierFitter().fit(T, E)
kmf.plot(figsize=(8,4))
plt.ylim(0, 1);
plt.title("Survival function estimated by KaplanMeier")

In [ ]:
naf = NelsonAalenFitter().fit(T, E)
naf.plot(figsize=(8,4))
plt.title("Cumulative hazard estimated by NelsonAalen")

In [ ]:
class UpperAsymptoteFitter(ParametricUnivariateFitter):

    _fitted_parameter_names = ["c_", "mu_", "sigma_"]

    _bounds = ((0, None), (None, None), (0, None))

    def _cumulative_hazard(self, params, times):
        c, mu, sigma = params
        return c * norm.cdf((times - mu) / sigma, loc=0, scale=1)

In [ ]:
uaf = UpperAsymptoteFitter().fit(T, E)
uaf.print_summary(3)
uaf.plot(figsize=(8,4))

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10, 6))

t = np.linspace(0, 40)
uaf = UpperAsymptoteFitter().fit(T, E, timeline=t)

uaf.plot_survival_function(ax=ax[0][0])
kmf.plot(ax=ax[0][0])

uaf.plot_cumulative_hazard(ax=ax[0][1])
naf.plot(ax=ax[0][1])

t = np.linspace(0, 100)
uaf = UpperAsymptoteFitter().fit(T, E, timeline=t)
uaf.plot_survival_function(ax=ax[1][0])
kmf.survival_function_.plot(ax=ax[1][0])

uaf.plot_cumulative_hazard(ax=ax[1][1])
naf.plot(ax=ax[1][1])

In [ ]:
from lifelines import AalenJohansenFitter
from lifelines.datasets import load_waltons
T, E = load_waltons()['T'], load_waltons()['E']
ajf = AalenJohansenFitter(calculate_variance=True)
ajf.fit(T, E, event_of_interest=1)
ajf.cumulative_density_
ajf.plot()